In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import mcstasscript as ms
import mcstastox as mx
import scipp as sc
import plopp as pp
from scippneutron.conversion.graph.beamline import beamline
from scippneutron.normalization import compute_q_de_norm

parent = os.path.dirname(os.getcwd())
sys.path.append(parent)

from trex_reduction import (
    generate_bins,
    inelastic,
    _calc_pulse_centroid,
)  # , produce_trex_event_object

ModuleNotFoundError: No module named 'trex_reduction'

In [ ]:
def produce_trex_event_object(
    event_object, data_path, monitor_name, centroids=None, to_s=1e-6
):
    """ """

    with mx.Read(data_path) as loaded_data:
        monitor_position = loaded_data.get_global_component_coordinates(monitor_name)

    data = ms.load_data(data_path)
    monitor = ms.name_search(monitor_name, data)

    event_object.coords["monitor_position"] = sc.vector(
        value=monitor_position, unit="m"
    )

    if centroids is None:
        centroids = _calc_pulse_centroid(monitor)

    look_up_tab = sc.DataArray(data=centroids, coords={"tof": centroids})

    tof_to_centroid = sc.lookup(look_up_tab, mode="previous")
    event_object = event_object.transform_coords(time_on_monitor=tof_to_centroid)
    event_object.coords["monitor_counts"] = sc.scalar(
        value=monitor.Ncount.sum(), unit="counts"
    )

    return event_object

In [ ]:
file_path = "/home/jl/Work/nscipp/data/LET_vanad"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp(
        source_name="SourceMantid",
        sample_name="iso_samp",
    )

data = ms.load_data(file_path)

events = scipp_object["events"]
events

In [ ]:
# McStas provides absolute time, not time of flight
events.bins.coords["tof"] = events.bins.coords["t"]
# Add additional information required for inelastic scattering
events = produce_trex_event_object(events, file_path, "Monitor6")
qens_graph = {**beamline(scatter=True), **inelastic}
events = events.transform_coords(["Q", "mag_Q", "dE"], graph=qens_graph)

# create (qx, qy, qz, en)
# Note: Need to use copy() to allow for binning later
events.bins.coords["qx"] = events.bins.coords["Q"].fields.x.copy()
events.bins.coords["qy"] = events.bins.coords["Q"].fields.y.copy()
events.bins.coords["qz"] = events.bins.coords["Q"].fields.z.copy()
events.bins.coords["en"] = events.bins.coords["dE"]

events

# Calculate minimum and maximum kf

In [ ]:
Lm = events.coords["Lm"]


def _calc_mag_kf_from_ef(ef):
    hbar = sc.constants.hbar
    m_n = sc.constants.neutron_mass
    kf = sc.sqrt(2 * m_n * ef) / hbar
    return kf.to(unit="1/angstrom")


monitor = ms.name_search("Monitor6", data)
tom_centroid = _calc_pulse_centroid(monitor)
vi = Lm / tom_centroid

ei = (0.5 * sc.constants.m_n * vi**2).to(unit="meV")

unit_ki = sc.vector([0, 0, 1])
mag_ki = ((sc.constants.neutron_mass * vi) / sc.constants.hbar).to(unit="1/angstrom")

ki = unit_ki * mag_ki

prop_ei = 0.8
max_ef = (1 + prop_ei) * ei
min_ef = (1 - prop_ei) * ei

min_kf = _calc_mag_kf_from_ef(min_ef)
max_kf = _calc_mag_kf_from_ef(max_ef)

print(f"min_kf: {min_kf}, max_kf: {max_kf}")

# Access and calculate detector trajectory endpoints

In [ ]:
sample_position = events.coords["sample_position"]
pixel_vec = scipp_object["positions"] - sample_position
pixel_vec = pixel_vec / sc.norm(pixel_vec)

Q_max = ki.to(unit="1/Å") - (pixel_vec * max_kf)
Q_min = ki.to(unit="1/Å") - (pixel_vec * min_kf)

kf_max = sc.broadcast(max_kf, dims=Q_max.dims, shape=Q_max.shape)
kf_min = sc.broadcast(min_kf, dims=Q_min.dims, shape=Q_min.shape)

kf_min

# Calculate detector solid angles dOmega (detector counts/monitor counts)

In [ ]:
scale_factor = (
    events.data.sum() / events.coords["monitor_counts"] * (4 * sc.constants.pi)
)
dOmega = (
    events.bins.sum().data / events.coords["monitor_counts"] * (4 * sc.constants.pi)
)
events.coords["dOmega"] = dOmega

# Generate 4D grid for histogramming

In [ ]:
bins = generate_bins(
    qx=(-2, 1.5, 0.1),
    qy=(-0.1, 0.1),
    qz=(-1, 3.0, 0.1),
    en=(-0.2, 0.2),
)

# convert energy to kf

In [ ]:
hbar = sc.constants.hbar
m_n = sc.constants.neutron_mass

kf_array = (
    (sc.sqrt(2 * m_n * (ei[0] - bins["en"])) / hbar)
    .to(unit="1/Å")
    .rename_dims({"en": "mag_kf"})
)
bins["mag_kf"] = kf_array[~sc.isnan(kf_array)]
bins["mag_kf"] = sc.sort(bins["mag_kf"], key="mag_kf")

edges = {key: bins[key] for key in ("qx", "qy", "qz", "mag_kf")}
display(edges)

# Calculate normalization factor per pixel

In [ ]:
hbar = sc.constants.hbar
m_n = sc.constants.neutron_mass


def _calc_en_from_kf_endpoints(kf_in, kf_out):
    en = hbar**2 / m_n * 0.5 * ((kf_in**2 - kf_out**2) * sc.Unit("1/Å^2"))
    return sc.abs(en).to(unit="meV")

In [ ]:
min_en = ei - min_ef
max_en = ei - max_ef
trajectory_start = (
    Q_min.squeeze().fields.x,
    Q_min.squeeze().fields.y,
    Q_min.squeeze().fields.z,
    min_en.broadcast(dims=Q_min.dims, shape=Q_min.shape).squeeze(),
)
trajectory_stop = (
    Q_max.squeeze().fields.x,
    Q_max.squeeze().fields.y,
    Q_max.squeeze().fields.z,
    max_en.broadcast(dims=Q_max.dims, shape=Q_max.shape).squeeze(),
)

grid = (
    edges["qx"],
    edges["qy"],
    edges["qz"],
    bins["en"].squeeze(),
)

norm = compute_q_de_norm(
    trajectory_start=tuple(zip(*trajectory_start)),
    trajectory_stop=tuple(zip(*trajectory_stop)),
    # The solid angle should be:
    # solid_angle=dOmega,
    # but dOmega is shorter than Q because it only has elements for pixels with events, not all detector pixels
    solid_angle=sc.ones(sizes={"pixel_id": Q_min.sizes["pixel_id"]}),
    grid=grid,
    incident_energy=ei.squeeze(),
)
norm = norm.rename_dims(h="qx", k="qy", l="qz", energy_transfer="en")

# Histogram data

In [ ]:
# %matplotlib widget

e = {
    "qx": edges["qx"],
    "qy": edges["qy"],
    "qz": edges["qz"],
    "en": bins["en"],
}
data_hist = sc.bin(events.data, **e).hist()
display(data_hist)
pp.plot(
    data_hist.squeeze().transpose(),
    coords=["qx", "qz"],
    grid=True,
    # cmap="turbo",
)

# Data divided by normalization factor

In [ ]:
data_hist.squeeze()

In [ ]:
norm

In [ ]:
# norm_factor = events.bins.concat().value.copy()
# norm_factor.data = sc.bins_like(events, events.coords["dOmega"]).bins.concat().value

# norm_hist = sc.bin(norm_factor, **edges).hist()

pp.plot(
    (data_hist.squeeze().transpose()) / norm.squeeze(),
    coords=["qx", "qz"],
    grid=True,
    # cmap="turbo",
    # vmin=0,
    # vmax=5e5,
)